In [4]:
import pandas as pd
import numpy as np

df_ligero = pd.read_parquet('traficoligero.parquet')
df_pesado = pd.read_parquet('traficopesado.parquet')

cols = ['Length', 'Inter-arrival']
df_ligero = df_ligero[cols].apply(pd.to_numeric, errors='coerce').dropna()
df_pesado = df_pesado[cols].apply(pd.to_numeric, errors='coerce').dropna()

df_ligero = df_ligero[(df_ligero['Inter-arrival'] > 0) & (df_ligero['Length'] > 0)]
df_pesado = df_pesado[(df_pesado['Inter-arrival'] > 0) & (df_pesado['Length'] > 0)]

centroide_ligero_len = df_ligero['Length'].mean()
centroide_ligero_int = df_ligero['Inter-arrival'].mean()

centroide_pesado_len = df_pesado['Length'].mean()
centroide_pesado_int = df_pesado['Inter-arrival'].mean()

min_len = min(df_ligero['Length'].min(), df_pesado['Length'].min())
max_len = max(df_ligero['Length'].max(), df_pesado['Length'].max())

df_todos_int = np.concatenate([df_ligero['Inter-arrival'].values, df_pesado['Inter-arrival'].values])
log_inter = np.log10(df_todos_int)
min_int = log_inter.min()
max_int = log_inter.max()

# Ligero
norm_lig_len = (centroide_ligero_len - min_len) / (max_len - min_len)
norm_lig_int = (np.log10(centroide_ligero_int) - min_int) / (max_int - min_int)

# Pesado
norm_pes_len = (centroide_pesado_len - min_len) / (max_len - min_len)
norm_pes_int = (np.log10(centroide_pesado_int) - min_int) / (max_int - min_int)

ideal_len = 1.0  # Paquete máximo
ideal_int = 0.0  # Tiempo de llegada 0
w_len = 2.0
w_int = 0.5

#Cálculo de Distancias Ponderadas (D)
D_ligero = w_len * abs(norm_lig_len - ideal_len) + w_int * abs(norm_lig_int - ideal_int)
D_pesado = w_len * abs(norm_pes_len - ideal_len) + w_int * abs(norm_pes_int - ideal_int)

print(f"Distancia del Modelo Ligero (D_ligero): {D_ligero:.4f}")
print(f"Distancia del Modelo Pesado (D_pesado): {D_pesado:.4f}")

if D_pesado < D_ligero:
    print("\nEl Tráfico Pesado es más representativo para el objetivo de saturación.")
else:
    print("\nEl Tráfico Ligero es más representativo para el objetivo de saturación.")

Distancia del Modelo Ligero (D_ligero): 2.2970
Distancia del Modelo Pesado (D_pesado): 2.2268

El Tráfico Pesado es más representativo para el objetivo de saturación.
